# 09 · RBIG (normalizing flow) por régimen

Ajusta un flujo normalizante por gaussianización iterativa, uno por régimen, con reducción previa de dimensión por PCA.

**Entradas**

- `data/processed/ventanas.npz`

**Salidas**

- `models/generadores/rbig/ (modelo.pkl o .keras, historial.csv, meta.json)`
- `data/synthetic/rbig.npz`

**Tiempo estimado:** ~15 min en CPU (hasta 40 capas por régimen, con parada por multi-información).

**Independencia.** Este notebook solo lee `data/processed/ventanas.npz` (notebook 02) y solo escribe en `models/generadores/rbig/` y `data/synthetic/rbig.npz`. No depende de ningún otro notebook de generador ni de sus salidas, de modo que los notebooks 04 a 10 pueden ejecutarse en paralelo y en cualquier orden por distintas personas.

In [ ]:
import sys; sys.path.insert(0, "..")   # permite ejecutar desde notebooks/
import src                              # fija el backend de Keras a PyTorch
from src import config, viz
config.fijar_semillas()
viz.aplicar_estilo()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import ventanas

part = ventanas.cargar_procesado()
train, val, test = part.train, part.val, part.test
print(train, val, test, sep="\n")

In [ ]:
from src import regimenes
from src.generadores import base

v = config.ventanas()
n_regimenes = config.n_regimenes()

bloque_train = ventanas.empaquetar(train)
print("bloque de train:", bloque_train.shape, "· d esperada:", ventanas.dimension_bloque(v))
regimenes.distribucion(train.y_reg, n_regimenes)

## Condicionamiento y reducción de dimensión

RBIG no admite condición: no hay ningún sitio donde inyectar la etiqueta. La
estrategia es ajustar un **modelo independiente por régimen** y elegir el que toque
al generar. Es la opción honesta —cada régimen conserva su propia estructura de
dependencia— pero traslada todo el peso al tamaño muestral: el régimen de crisis
puede tener unos cientos de ventanas, y ahí la CDF empírica se estima con muy poca
información.

El bloque tiene 1.201 dimensiones. Una PCA de rotación en ese espacio con `n < d` es
degenerada: la covarianza no tiene rango completo y las direcciones sobrantes son
ruido puro. Por eso se reduce primero a `n_componentes`, se ajusta RBIG en el
espacio reducido y al generar se proyecta de vuelta.

El precio es explícito: las muestras viven en un subespacio afín de dimensión `k` y
la varianza descartada se pierde. Con `ruido_residual=True` esa varianza se repone
como ruido gaussiano independiente por dimensión, para que la covarianza sintética
no sea exactamente singular.

In [ ]:
generador = base.instanciar(
    "rbig",
    n_regimenes=n_regimenes,
    n_componentes=128,
    max_capas=40,
    min_capas=3,
    umbral_info=0.01,
    paciencia=3,
    ruido_residual=True,
    verboso=True,
)
generador.fit(bloque_train, train.y_reg)
generador

## Convergencia

No hay curva de pérdida: nada se optimiza por gradiente. El equivalente en RBIG es
la **reducción de multi-información por capa**, que mide cuánta dependencia
estadística ha destruido cada capa.

La lectura es la de una loss con el signo invertido: la curva acumulada crece rápido
en las primeras capas y se aplana cuando ya no queda dependencia que extraer. Ese
aplanamiento es el criterio de parada; seguir añadiendo capas a partir de ahí solo
memoriza los cuantiles de la muestra de train.

La reducción registrada está corregida por el sesgo del estimador, sin lo cual la
curva no se aplanaría nunca con los tamaños muestrales de este taller.

El historial concatena los tres regímenes —cada uno reinicia el contador de capas—,
así que se grafica un panel por régimen en lugar de una curva única.

In [ ]:
fig, ejes = plt.subplots(1, n_regimenes, figsize=(4.2 * n_regimenes, 3.8), sharex=True)
nombres = ["calma", "transición", "crisis"][:n_regimenes]

for k, eje in enumerate(np.atleast_1d(ejes)):
    tramo = generador.historial[generador.historial["regimen"] == k]
    if tramo.empty:
        eje.set_visible(False)
        continue
    viz.curva_convergencia(
        tramo.drop(columns="regimen"),
        "RBIG · régimen {} ({})".format(k, nombres[k]),
        eje=eje,
    )
    eje.set_xlabel("capa")
    eje.set_ylabel("nats")

fig.tight_layout()
viz.guardar(fig, "convergencia_rbig")

generador.historial.groupby("regimen").agg(
    capas=("epoca", "max"), reduccion_final=("reduccion_acumulada", "last")
)

## Inspección visual

Proyección PCA de reales y sintéticos, con la PCA ajustada **solo con los reales**
para que los ejes describan la estructura del mercado y no la del generador.

Es la comprobación más rápida y la que detecta los dos fallos gruesos: si la nube
sintética no cubre la real, el generador ha colapsado a un modo; si la desborda
ampliamente, está inventando configuraciones de mercado que nunca ocurrieron.

Se mira el régimen de crisis porque es el que tiene menos datos reales y, por
tanto, donde el generador tiene más margen para desviarse.

In [ ]:
CRISIS = n_regimenes - 1

muestra_crisis = generador.generate(600, regimen=CRISIS)
reales_crisis = bloque_train[train.y_reg == CRISIS]

fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))
viz.real_vs_sintetico(bloque_train, generador.generate(600, regimen=0),
                      "{} · régimen de calma".format(generador.etiqueta), eje=ejes[0])
viz.real_vs_sintetico(reales_crisis, muestra_crisis,
                      "{} · régimen de crisis".format(generador.etiqueta), eje=ejes[1])
fig.tight_layout()
viz.guardar(fig, "pca_" + generador.nombre)

print("reales de crisis:", len(reales_crisis), "· sintéticos generados:", len(muestra_crisis))

## Banco de muestras

Se genera un banco uniforme por régimen y se exporta a `data/synthetic/`. La mezcla
concreta de cada dataset la decide el notebook 11 muestreando de este banco, no
volviendo a invocar al generador: así el barrido no necesita tener los siete
modelos cargados en memoria y dos ejecuciones del notebook 12 usan exactamente las
mismas muestras sintéticas.

El banco es uniforme —no replica el desbalance real— porque la política de reparto
es un grado de libertad del experimento y se aplica después.

In [ ]:
MUESTRAS_POR_REGIMEN = 3000

reparto = {k: MUESTRAS_POR_REGIMEN for k in range(n_regimenes)}
bloques_sint, y_sint = generador.generate_dataset(reparto)

print("banco:", bloques_sint.shape, "· etiquetas:", np.bincount(y_sint, minlength=n_regimenes))
print("rango de valores:", round(float(bloques_sint.min()), 2), "→",
      round(float(bloques_sint.max()), 2),
      "(referencia real:", round(float(bloque_train.min()), 2), "→",
      round(float(bloque_train.max()), 2), ")")

## Persistencia

`guardar()` deja el modelo, la curva de convergencia y los metadatos en
`models/generadores/`. Es lo que permite que el resto del grupo salte directamente
al análisis sin reentrenar nada.

In [ ]:
ruta_muestras = generador.exportar_muestras(bloques_sint, y_sint)
ruta_modelo = generador.guardar()

print("muestras:", ruta_muestras)
print("modelo:  ", ruta_modelo)
pd.Series(generador.resumen_convergencia()).round(4)

## Salidas generadas

In [ ]:
from pathlib import Path

salidas = [
    src.DIR_MODELOS_GEN / "rbig" / "meta.json",
    src.DIR_MODELOS_GEN / "rbig" / "historial.csv",
    src.DIR_SINTETICO / "rbig.npz",
    src.DIR_FIGURAS / "convergencia_rbig.png",
    src.DIR_FIGURAS / "pca_rbig.png",
]

for ruta in salidas:
    ruta = Path(ruta)
    marca = "ok" if ruta.exists() else "--"
    print("[{}] {}".format(marca, ruta.relative_to(src.RAIZ)))
